In [1]:
import meshio
import numpy as np
# from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from scipy.spatial import cKDTree


FEMMESH = meshio.read("wing.msh")
FEMPOINT = FEMMESH.points * 1E-3
FEMTRIANGLE = FEMMESH.cells_dict["triangle"]

bbox = np.ptp(FEMPOINT, axis=0)
print("bbox fem =", bbox)

FOAMMESH = meshio.read("FOAMData.xdmf")
FOAMPOINT = FOAMMESH.points
FOAMTRIANGLE = FOAMMESH.cells_dict["triangle"]
FOAMTRACTION = FOAMMESH.point_data["traction"] 

bbox = np.ptp(FOAMPOINT, axis=0)
print("bbox foam =", bbox)

FOAM_FORCE = np.zeros(3)

for TRI in FOAMTRIANGLE:
  I0, I1, I2 = TRI
  P0 = FOAMPOINT[I0]
  P1 = FOAMPOINT[I1]
  P2 = FOAMPOINT[I2]
  AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
  AVG_FOAMTRACTION = (
    FOAMTRACTION[I0] +
    FOAMTRACTION[I1] +
    FOAMTRACTION[I2]
  ) / 3.0
  FOAM_FORCE += AVG_FOAMTRACTION * AREA

print("FOAM integrated force [N]:", FOAM_FORCE)

# interp_x = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 0])
# interp_y = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 1])
# interp_z = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 2])

# FEMTRACTION = np.column_stack([
#   interp_x(FEMPOINT),
#   interp_y(FEMPOINT),
#   interp_z(FEMPOINT)
# ])

TREE = cKDTree(FOAMPOINT)
DIST, ID = TREE.query(FEMPOINT)
FEMTRACTION = FOAMTRACTION[ID]

# print("start lump force from distributed traction")
NUM_POINT = len(FEMPOINT)
print(NUM_POINT)
NODAL_FORCE = np.zeros((NUM_POINT, 3))
FEM_FORCE = np.zeros(3)

for TRI in FEMTRIANGLE:
  I0, I1, I2 = TRI
  P0 = FEMPOINT[I0]
  P1 = FEMPOINT[I1]
  P2 = FEMPOINT[I2]
  AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
#   AVG_TRACTION = FEMTRACTION[TRI].mean(axis=0)
#   AVG_FORCE = AVG_TRACTION * AREA
#   for i in TRI:
#     NODAL_FORCE[i] += AVG_FORCE / 3.0
  AVG_TRACTION = (FEMTRACTION[I0] + FEMTRACTION[I1] + FEMTRACTION[I2]) / 3.0
  FEM_FORCE += AVG_TRACTION * AREA

print("FEM integrated force [N]:", FEM_FORCE)

# TOTAL_SURFACE_FORCE = np.sum(NODAL_FORCE, axis=0)
# print("Total integrated force [N]:", TOTAL_SURFACE_FORCE)
# print("end lump force from distributed traction")

error = np.linalg.norm(FEM_FORCE - FOAM_FORCE) / np.linalg.norm(FOAM_FORCE)

print("Relative force error:", error)

NODAL_FORCE = np.zeros((NUM_POINT, 3))

for TRI in FEMTRIANGLE:
    I0, I1, I2 = TRI
    P0, P1, P2 = FEMPOINT[I0], FEMPOINT[I1], FEMPOINT[I2]
    AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
    AVG_TRACTION = (FEMTRACTION[I0] + FEMTRACTION[I1] + FEMTRACTION[I2]) / 3.0
    CONTRIBUTION = AVG_TRACTION * AREA / 3.0
    NODAL_FORCE[I0] += CONTRIBUTION
    NODAL_FORCE[I1] += CONTRIBUTION
    NODAL_FORCE[I2] += CONTRIBUTION

# Verification
TOTAL = np.sum(NODAL_FORCE, axis=0)
print("Total lumped nodal force [N]:", TOTAL)
print("FOAM reference force    [N]:", FOAM_FORCE)
print("Relative error:", np.linalg.norm(TOTAL - FOAM_FORCE) / np.linalg.norm(FOAM_FORCE))


bbox fem = [2.499403  3.        0.1222867]
bbox foam = [2.4905794  3.0000658  0.12269325]
FOAM integrated force [N]: [  184.69715373 -4294.18893721 17846.43225416]
3280
FEM integrated force [N]: [-1255.27477251 -4065.82754782 18354.57942287]
Relative force error: 0.08410990922709234
Total lumped nodal force [N]: [-1255.27477251 -4065.82754782 18354.57942287]
FOAM reference force    [N]: [  184.69715373 -4294.18893721 17846.43225416]
Relative error: 0.08410990922709927


In [2]:
# Thay vì map point-by-point, chỉ cần tổng lực đúng
# Scale FEMTRACTION để tổng lực khớp với FOAM

# Tính tổng lực FEM hiện tại theo từng axis
SCALE = FOAM_FORCE / (FEM_FORCE + 1e-12)
print("Scale factors:", SCALE)

# Scale nodal force
NODAL_FORCE_CORRECTED = NODAL_FORCE * SCALE[np.newaxis, :]
print("Corrected total:", np.sum(NODAL_FORCE_CORRECTED, axis=0))

Scale factors: [-0.14713683  1.05616603  0.97231497]
Corrected total: [  184.69715373 -4294.18893721 17846.43225416]


In [3]:
from scipy.spatial import cKDTree

# Dùng centroid của FOAM triangles thay vì points
FOAM_CENTROIDS = FOAMPOINT[FOAMTRIANGLE].mean(axis=1)
FOAM_TRACTION_CENTROIDS = FOAMTRACTION[FOAMTRIANGLE].mean(axis=1)

TREE = cKDTree(FOAM_CENTROIDS)

NODAL_FORCE = np.zeros((NUM_POINT, 3))
FEM_FORCE = np.zeros(3)

for TRI in FEMTRIANGLE:
    I0, I1, I2 = TRI
    P0, P1, P2 = FEMPOINT[I0], FEMPOINT[I1], FEMPOINT[I2]
    CENTROID_FEM = (P0 + P1 + P2) / 3.0
    AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
    
    # Tìm FOAM triangle gần nhất theo centroid
    _, TID = TREE.query(CENTROID_FEM)
    TRACTION = FOAM_TRACTION_CENTROIDS[TID]
    
    CONTRIBUTION = TRACTION * AREA / 3.0
    NODAL_FORCE[I0] += CONTRIBUTION
    NODAL_FORCE[I1] += CONTRIBUTION
    NODAL_FORCE[I2] += CONTRIBUTION
    FEM_FORCE += TRACTION * AREA

print("FOAM integrated force [N]:", FOAM_FORCE)
print("FEM  integrated force [N]:", FEM_FORCE)
error = np.linalg.norm(FEM_FORCE - FOAM_FORCE) / np.linalg.norm(FOAM_FORCE)
print("Relative force error:", error)

FOAM integrated force [N]: [  184.69715373 -4294.18893721 17846.43225416]
FEM  integrated force [N]: [ -329.38414324 -2964.39909531 17749.97118196]
Relative force error: 0.07784392964439805


In [5]:
import meshio
import numpy as np
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
# from scipy.spatial import cKDTree


FEMMESH = meshio.read("wing.msh")
FEMPOINT = FEMMESH.points * 1E-3
FEMTRIANGLE = FEMMESH.cells_dict["triangle"]

FOAMMESH = meshio.read("FOAMData.xdmf")
FOAMPOINT = FOAMMESH.points
FOAMTRIANGLE = FOAMMESH.cells_dict["triangle"]
FOAMTRACTION = FOAMMESH.point_data["traction"] 

bboxfoam = np.ptp(FOAMPOINT, axis=0)
bboxfem  = np.ptp(FEMPOINT, axis=0)
print("bbox fem  =", bboxfoam)
print("bbox foam =", bboxfem)

interp_x = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 0])
interp_y = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 1])
interp_z = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 2])

FEMTRACTION = np.column_stack([
  interp_x(FEMPOINT),
  interp_y(FEMPOINT),
  interp_z(FEMPOINT)
])



print("start lump force from distributed traction")
NUM_POINT = len(FEMPOINT)
print(NUM_POINT)
NODAL_FORCE = np.zeros((NUM_POINT, 3))

for TRI in FEMTRIANGLE:
  I0, I1, I2 = TRI
  P0 = FEMPOINT[I0]
  P1 = FEMPOINT[I1]
  P2 = FEMPOINT[I2]
  AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
  AVG_TRACTION = 
  AVG_TRACTION = (FEMTRACTION[I0] + FEMTRACTION[I1] + FEMTRACTION[I2]) / 3.0
  FEM_FORCE += AVG_TRACTION * AREA

FEM_FORCE = np.zeros(3)

print("FEM integrated force [N]:", FEM_FORCE)

error = np.linalg.norm(FEM_FORCE - FOAM_FORCE) / np.linalg.norm(FOAM_FORCE)

print("Relative force error:", error)


bbox fem = [2.499403  3.        0.1222867]
bbox foam = [2.4905794  3.0000658  0.12269325]
FOAM integrated force [N]: [  184.69715373 -4294.18893721 17846.43225416]
3280
FEM integrated force [N]: [-1255.2747731  -4065.82752961 18354.57940929]
Relative force error: 0.0841099091602138


In [14]:
import meshio
import numpy as np
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

FEMMESH = meshio.read("wing.msh")
FEMPOINT = FEMMESH.points * 1E-3
FEMTRIANGLE = FEMMESH.cells_dict["triangle"]

FOAMMESH = meshio.read("FOAMData.xdmf")
FOAMPOINT = FOAMMESH.points
FOAMTRIANGLE = FOAMMESH.cells_dict["triangle"]
FOAMTRACTION = FOAMMESH.point_data["traction"] 

bboxfoam = np.ptp(FOAMPOINT, axis=0)
bboxfem  = np.ptp(FEMPOINT, axis=0)
print("bbox fem  =", bboxfoam)
print("bbox foam =", bboxfem)

interp_x = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 0])
interp_y = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 1])
interp_z = NearestNDInterpolator(FOAMPOINT, FOAMTRACTION[:, 2])

FEMTRACTION = np.column_stack([
  interp_x(FEMPOINT),
  interp_y(FEMPOINT),
  interp_z(FEMPOINT)
])

MAPPEDTRACTION = meshio.Mesh(
  points=FEMPOINT,
  cells=[("triangle",FEMMESH.cells_dict["triangle"])],
  point_data={"traction": FEMTRACTION}
)
meshio.write("MappedTraction.xdmf", MAPPEDTRACTION)



print("start lump force from distributed traction")
MESH = meshio.read("MappedTraction.xdmf")
POINT = MESH.points

NUM_POINT = len(POINT)
NODAL_FORCE = np.zeros((NUM_POINT, 3))
TRIANGLE = MESH.cells_dict["triangle"]
TRACTION = MESH.point_data["traction"]

for TRI in TRIANGLE:
  P0, P1, P2 = POINT[TRI]
  AREA = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
  AVG_TRACTION = TRACTION[TRI].mean(axis=0)
  TRIANGLE_FORCE = AREA * AVG_TRACTION
  for i in TRI:
    NODAL_FORCE[i] += TRIANGLE_FORCE / 3.0

print("start lump force from distributed traction")

FEM_FORCE = np.sum(NODAL_FORCE, axis=0)

print("FEM integrated force [N]:", FEM_FORCE)



FOAMP       = FOAMMESH.point_data["p"]
FOAMNORMALS = FOAMMESH.point_data["normals"]
FOAMWSS     = FOAMMESH.point_data["wss"]

PRESSURE_FORCE = np.zeros(3)
VISCOUS_FORCE  = np.zeros(3)

for TRI in FOAMTRIANGLE:
    I0, I1, I2 = TRI
    P0, P1, P2 = FOAMPOINT[I0], FOAMPOINT[I1], FOAMPOINT[I2]
    AREA = 0.5 * np.linalg.norm(np.cross(P1-P0, P2-P0))
    
    avg_p   = (FOAMP[I0]       + FOAMP[I1]       + FOAMP[I2])       / 3.0
    avg_n   = (FOAMNORMALS[I0] + FOAMNORMALS[I1] + FOAMNORMALS[I2]) / 3.0
    avg_wss = (FOAMWSS[I0]     + FOAMWSS[I1]     + FOAMWSS[I2])     / 3.0

    PRESSURE_FORCE += -avg_p * avg_n * AREA
    VISCOUS_FORCE  += avg_wss * AREA

print("Pressure force [N]:", PRESSURE_FORCE)
print("Viscous  force [N]:", VISCOUS_FORCE)
print("Total    force [N]:", PRESSURE_FORCE + VISCOUS_FORCE)


bbox fem  = [2.4905794  3.0000658  0.12269325]
bbox foam = [2.499403  3.        0.1222867]
start lump force from distributed traction
start lump force from distributed traction
FEM integrated force [N]: [   719.92773895   4063.45350718 -18333.15928655]
Pressure force [N]: [  -435.3215969    4283.02641702 -17810.53860153]
Viscous  force [N]: [-266.99168257   -0.97437948   10.57383665]
Total    force [N]: [  -702.31327947   4282.05203755 -17799.96476488]


In [15]:
# Tích phân traction trực tiếp trên FOAM mesh (ground truth)
FOAM_FORCE_CHECK = np.zeros(3)
for TRI in FOAMTRIANGLE:
    I0, I1, I2 = TRI
    P0, P1, P2 = FOAMPOINT[I0], FOAMPOINT[I1], FOAMPOINT[I2]
    AREA = 0.5 * np.linalg.norm(np.cross(P1-P0, P2-P0))
    AVG  = (FOAMTRACTION[I0] + FOAMTRACTION[I1] + FOAMTRACTION[I2]) / 3.0
    FOAM_FORCE_CHECK += AVG * AREA

print("FOAM self-check  [N]:", FOAM_FORCE_CHECK)
print("FOAM reference   [N]: [1133, -4433, 17825]")
print("FEM mapped force [N]:", FEM_FORCE)

FOAM self-check  [N]: [  -718.68051629   4292.24017148 -17825.28454839]
FOAM reference   [N]: [1133, -4433, 17825]
FEM mapped force [N]: [   719.92773895   4063.45350718 -18333.15928655]
